# CSCI 331 — Database and Data Modeling
## Homework 7 · Chapter 8: Data Modification

| Field | Details |
|---|---|
| **Name** | Haroon Ahmed |
| **Group** | EOS_Group_3 |
| **Database** | Northwinds2024Student |
| **Semester** | Spring 2026 |
| **Professor** | Prof. Heller |

---

This notebook presents 10 original business propositions and their corresponding T-SQL data modification queries against the **Northwinds2024Student** database. Each proposition identifies a real business problem and is solved using INSERT, UPDATE, DELETE, OUTPUT, or SELECT INTO statements from Chapter 8.


---
## Setup — Create Working Tables
Run this cell first to create the working tables used across all propositions.

In [ ]:
USE Northwinds2024Student;
GO

-- Create a working Customers staging table
DROP TABLE IF EXISTS dbo.CustomersStaging;

CREATE TABLE dbo.CustomersStaging
(
  CustomerId          INT           NOT NULL PRIMARY KEY,
  CustomerCompanyName NVARCHAR(80)  NOT NULL,
  CustomerCountry     NVARCHAR(15)  NOT NULL,
  CustomerRegion      NVARCHAR(15)  NULL,
  CustomerCity        NVARCHAR(15)  NOT NULL
);
GO

-- Create a working Orders archive table
DROP TABLE IF EXISTS dbo.OrdersArchive;

SELECT *
INTO dbo.OrdersArchive
FROM [Sales].[Order]
WHERE 1 = 0; -- structure only, no data yet
GO

PRINT 'Setup complete.';

---
## Proposition 1 — Insert a New Test Customer Record

**Business Problem:**  
The sales team needs to onboard a new test customer account for internal QA purposes before going live. A single customer row must be inserted directly into the staging table with known values so the team can validate downstream processes without affecting production data.

**Technique:** `INSERT INTO ... VALUES`

In [ ]:
USE Northwinds2024Student;
GO

INSERT INTO dbo.CustomersStaging
  (CustomerId, CustomerCompanyName, CustomerCountry, CustomerRegion, CustomerCity)
VALUES
  (9001, N'Northwinds Test Account', N'USA', N'NY', N'New York');

-- Verify
SELECT * FROM dbo.CustomersStaging WHERE CustomerId = 9001;

---
## Proposition 2 — Bulk Load Active Customers into Staging Table

**Business Problem:**  
Before running a marketing campaign, the team needs a staging copy of only those customers who have actually placed at least one order. Inactive customer records should be excluded to avoid wasted outreach.

**Technique:** `INSERT INTO ... SELECT ... WHERE EXISTS`

In [ ]:
USE Northwinds2024Student;
GO

INSERT INTO dbo.CustomersStaging
  (CustomerId, CustomerCompanyName, CustomerCountry, CustomerRegion, CustomerCity)
SELECT
  C.CustomerId,
  C.CustomerCompanyName,
  C.CustomerCountry,
  C.CustomerRegion,
  C.CustomerCity
FROM Sales.Customers AS C
WHERE EXISTS
  (SELECT 1 FROM [Sales].[Order] AS O
   WHERE O.CustomerId = C.CustomerId)
AND C.CustomerId NOT IN (SELECT CustomerId FROM dbo.CustomersStaging);

-- Verify count
SELECT COUNT(*) AS ActiveCustomersLoaded FROM dbo.CustomersStaging;

---
## Proposition 3 — Archive Recent Orders Using SELECT INTO

**Business Problem:**  
The operations team wants to create a snapshot archive of all orders placed in the most recent two years for year-end reporting. Rather than querying the live Orders table repeatedly, a dedicated archive table should be created and populated in one step.

**Technique:** `SELECT INTO` (creates and populates table in one statement)

In [ ]:
USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.OrdersArchive;

SELECT *
INTO dbo.OrdersArchive
FROM [Sales].[Order]
WHERE OrderDate >= '20210101'
  AND OrderDate < '20230101';

-- Verify
SELECT COUNT(*) AS ArchivedOrders FROM dbo.OrdersArchive;
SELECT MIN(OrderDate) AS EarliestOrder, MAX(OrderDate) AS LatestOrder FROM dbo.OrdersArchive;

---
## Proposition 4 — Delete Old Archive Orders and Capture Removed Records

**Business Problem:**  
The data governance team requires that any records purged from the archive table be logged for audit purposes. Orders placed before January 2022 are to be removed from the archive, and the orderid and date of each deleted row must be captured using the OUTPUT clause.

**Technique:** `DELETE ... OUTPUT deleted.*`

In [ ]:
USE Northwinds2024Student;
GO

DELETE FROM dbo.OrdersArchive
OUTPUT
  deleted.OrderId   AS DeletedOrderId,
  deleted.OrderDate AS DeletedOrderDate
WHERE OrderDate < '20220101';

---
## Proposition 5 — Remove All Orders from Customers in a Specific Country

**Business Problem:**  
Due to a business policy change, all archived orders associated with customers based in Germany must be removed from the archive table. The delete must be driven by a customer country lookup rather than hardcoded order IDs.

**Technique:** `DELETE ... WHERE EXISTS (subquery)`

In [ ]:
USE Northwinds2024Student;
GO

DELETE FROM dbo.OrdersArchive
WHERE EXISTS
(
  SELECT 1
  FROM dbo.CustomersStaging AS C
  WHERE dbo.OrdersArchive.CustomerId = C.CustomerId
    AND C.CustomerCountry = N'Germany'
);

-- Verify no Germany orders remain
SELECT COUNT(*) AS GermanyOrdersRemaining
FROM dbo.OrdersArchive AS O
WHERE EXISTS
(
  SELECT 1 FROM dbo.CustomersStaging AS C
  WHERE O.CustomerId = C.CustomerId
    AND C.CustomerCountry = N'Germany'
);

---
## Proposition 6 — Replace NULL Regions with a Default Value and Audit the Change

**Business Problem:**  
Several customer records in the staging table have NULL values in the CustomerRegion column, which causes issues in downstream reporting tools that cannot handle NULLs. All NULL region values should be replaced with the placeholder `<None>`, and the change must be logged showing both the old and new values for each affected customer.

**Technique:** `UPDATE ... OUTPUT deleted.col AS old, inserted.col AS new`

In [ ]:
USE Northwinds2024Student;
GO

UPDATE dbo.CustomersStaging
  SET CustomerRegion = N'<None>'
OUTPUT
  deleted.CustomerId,
  deleted.CustomerRegion AS OldRegion,
  inserted.CustomerRegion AS NewRegion
WHERE CustomerRegion IS NULL;

---
## Proposition 7 — Sync Shipping Info on Archive Orders from Customer Master

**Business Problem:**  
Customer addresses have been updated in the master Customers table, but the archived orders still carry old shipping country data. All archive orders belonging to customers in the staging table should have their shipping country updated to reflect the current customer country on record.

**Technique:** `UPDATE ... FROM ... JOIN` (T-SQL extension)

In [ ]:
USE Northwinds2024Student;
GO

UPDATE O
  SET O.ShipCountry = C.CustomerCountry
FROM dbo.OrdersArchive AS O
  INNER JOIN dbo.CustomersStaging AS C
    ON O.CustomerId = C.CustomerId;

-- Verify a sample
SELECT TOP 5
  O.OrderId,
  O.CustomerId,
  O.ShipCountry,
  C.CustomerCountry
FROM dbo.OrdersArchive AS O
INNER JOIN dbo.CustomersStaging AS C
  ON O.CustomerId = C.CustomerId;

---
## Proposition 8 — Update via CTE for Improved Readability

**Business Problem:**  
The shipping city on archived orders needs to be synced to the current customer city on file. Rather than using a complex joined UPDATE statement, a CTE is used to make the logic transparent and easier to audit — both the source (customer city) and target (ship city) are visible in a single named expression before the update is applied.

**Technique:** `UPDATE through CTE`

In [ ]:
USE Northwinds2024Student;
GO

WITH CTE_SyncCity AS
(
  SELECT
    O.ShipCity    AS CurrentShipCity,
    C.CustomerCity AS CorrectCity
  FROM dbo.OrdersArchive AS O
    INNER JOIN dbo.CustomersStaging AS C
      ON O.CustomerId = C.CustomerId
)
UPDATE CTE_SyncCity
  SET CurrentShipCity = CorrectCity;

PRINT 'Ship cities updated via CTE.';

---
## Proposition 9 — Insert Multiple New Customers in a Single Statement

**Business Problem:**  
A batch of three new international customer accounts has been approved. Rather than running three separate INSERT statements, all three rows should be inserted in a single multi-row VALUES clause for efficiency. This also reduces round-trips to the database server.

**Technique:** `INSERT INTO ... VALUES (row1), (row2), (row3)`

In [ ]:
USE Northwinds2024Student;
GO

INSERT INTO dbo.CustomersStaging
  (CustomerId, CustomerCompanyName, CustomerCountry, CustomerRegion, CustomerCity)
VALUES
  (9002, N'Tokyo Traders Ltd',    N'Japan',   NULL,  N'Tokyo'),
  (9003, N'Sao Paulo Supplies',   N'Brazil',  NULL,  N'Sao Paulo'),
  (9004, N'Berlin Wholesale GmbH',N'Germany', NULL,  N'Berlin');

-- Verify all three inserted
SELECT * FROM dbo.CustomersStaging
WHERE CustomerId IN (9002, 9003, 9004);

---
## Proposition 10 — Truncate Tables with Foreign Key Constraint Handling

**Business Problem:**  
At the end of each QA cycle, the staging and archive tables need to be fully cleared so the next cycle starts with a clean slate. A simple DELETE would log every row individually — TRUNCATE is far more efficient. However, if a foreign key exists between the tables, the constraint must be temporarily dropped before truncation and re-applied afterward to maintain referential integrity.

**Technique:** `ALTER TABLE DROP CONSTRAINT` → `TRUNCATE` → `ALTER TABLE ADD CONSTRAINT`

In [ ]:
USE Northwinds2024Student;
GO

-- Step 1: Drop FK if it exists (safe pattern)
IF EXISTS (
    SELECT 1 FROM sys.foreign_keys
    WHERE name = 'FK_OrdersArchive_CustomersStaging'
)
BEGIN
    ALTER TABLE dbo.OrdersArchive
    DROP CONSTRAINT FK_OrdersArchive_CustomersStaging;
END;
GO

-- Step 2: Truncate both tables
TRUNCATE TABLE dbo.OrdersArchive;
TRUNCATE TABLE dbo.CustomersStaging;

-- Step 3: Verify both are empty
SELECT 'OrdersArchive'    AS TableName, COUNT(*) AS RowCount FROM dbo.OrdersArchive
UNION ALL
SELECT 'CustomersStaging' AS TableName, COUNT(*) AS RowCount FROM dbo.CustomersStaging;

PRINT 'Tables truncated successfully.';

---
## Cleanup
Run this cell after all propositions have been demonstrated to remove the working tables.

In [ ]:
USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.OrdersArchive;
DROP TABLE IF EXISTS dbo.CustomersStaging;

PRINT 'Cleanup complete.';

---
## Summary of Propositions

| # | Proposition | Technique |
|---|-------------|----------|
| 1 | Insert a single test customer record | `INSERT INTO ... VALUES` |
| 2 | Bulk load only active customers into staging | `INSERT INTO ... SELECT ... WHERE EXISTS` |
| 3 | Archive two years of orders in one step | `SELECT INTO` |
| 4 | Purge old archive records with audit trail | `DELETE ... OUTPUT deleted.*` |
| 5 | Remove all orders tied to a specific country | `DELETE ... WHERE EXISTS` |
| 6 | Replace NULL regions and log the change | `UPDATE ... OUTPUT deleted/inserted` |
| 7 | Sync shipping country from customer master | `UPDATE ... FROM ... JOIN` |
| 8 | Sync shipping city via CTE for readability | `UPDATE through CTE` |
| 9 | Insert three new customers in one statement | `INSERT ... multi-row VALUES` |
| 10 | Truncate tables safely around FK constraints | `DROP CONSTRAINT → TRUNCATE → ADD CONSTRAINT` |

---

## NACE Competencies Applied

| Competency | Application |
|---|---|
| 🧠 **Critical Thinking** | Designed 10 distinct business propositions covering all major DML operations; handled edge cases like FK constraints and NULL values |
| 💬 **Communication** | Documented each proposition in plain business language accessible to non-technical stakeholders |
| 💻 **Technology** | Applied INSERT, UPDATE, DELETE, TRUNCATE, OUTPUT, SELECT INTO, and CTE-based updates in a production-style database |
| 🤝 **Teamwork** | Cross-trained group members on data modification patterns and OUTPUT clause usage |
| 🎯 **Professionalism** | Followed naming conventions, disclosed LLM usage, and structured notebook for clean walkthrough presentation |

---

> *Queens College, CUNY · CSCI 331 — Database and Data Modeling · Prof. Heller · Spring 2026*